In [14]:
from future_rag.indexing.embeddings import EmbeddingFactory
from future_rag.indexing.indexer import IndexBuilder
from future_rag.indexing.loader import IndexLoader
from future_rag.indexing.vectorstore import VectorStore
from future_rag.knowledge.chunker import DocumentChunker
from future_rag.knowledge.loaders import DocumentLoader
from future_rag.llms.deepseek import DeepSeekLLM
from future_rag.pipelines.basic_rag import BasicRAG
from future_rag.retrieval.retriever import Retriever
from future_rag.config.settings import RAGConfig

from pathlib import Path
from dotenv import load_dotenv

import os
import warnings

load_dotenv()
config = RAGConfig()
PROJECT_ROOT = Path.cwd().parent

warnings.filterwarnings("ignore")

In [17]:
# ============================================================
# 1. Load documents
#    Read source documents and convert them into LlamaIndex Documents
# ============================================================



pdf_path = PROJECT_ROOT / config.documents_path

print("=" * 70)
print("📄 Loading Documents")
print("=" * 70)

print(f"📂 Source path: {pdf_path}")

documents = DocumentLoader.load(pdf_path)

print(f"✅ Loaded {len(documents)} document(s).")


# ============================================================
# 2. Split documents into chunks
#    Convert Documents into smaller Nodes for embedding/retrieval
# ============================================================

print("\n" + "=" * 70)
print("✂️  Chunking Documents")
print("=" * 70)

chunk_size = config.chunk_size
chunk_overlap = config.chunk_overlap

print(f"⚙️ Chunk size   : {chunk_size}")
print(f"⚙️ Chunk overlap: {chunk_overlap}")

chunker = DocumentChunker(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
)

nodes = chunker.split(documents)

print(f"✅ Generated {len(nodes)} chunk(s)/node(s).")


# ============================================================
# 3. Initialize embedding model
#    Create embeddings used for indexing and retrieval
# ============================================================

print("\n" + "=" * 70)
print("🧠 Initializing Embedding Model")
print("=" * 70)

embedding_model = EmbeddingFactory.create(config.embedding_model)

print(f"✅ Embedding model ready: {type(embedding_model).__name__}")


# ============================================================
# 4. Connect to vector database
#    Create or load ChromaDB persistent vector store
# ============================================================

print("\n" + "=" * 70)
print("🗄️  Connecting to ChromaDB")
print("=" * 70)

db_path = PROJECT_ROOT / config.vectorstore_path
collection_name = config.collection_name

print(f"📂 Database path : {db_path}")
print(f"📌 Collection    : {collection_name}")


vector_store_manager = VectorStore(
    db_path=db_path,
    collection_name=collection_name,
)

chroma_vector_store = vector_store_manager.connect()

print("✅ Connected to ChromaVectorStore successfully!")


# ============================================================
# 5. Create or load index
#    Reuse existing vectors or build a new index
# ============================================================

print("\n" + "=" * 70)
print("📚 Preparing Vector Index")
print("=" * 70)


existing_vector_count = chroma_vector_store._collection.count()

if existing_vector_count > 0:

    print(
        f"⚠️ Existing index found with {existing_vector_count} vector(s)."
    )

    print("🔄 Loading existing index using IndexLoader...")

    loader = IndexLoader(
        embedding_model=embedding_model,
        vector_store=chroma_vector_store,
    )

    index = loader.load()

    print("✅ Existing index loaded successfully.")

else:

    print("ℹ️ No existing vectors found.")
    print("🏗️ Building new index using IndexBuilder...")

    builder = IndexBuilder(
        embedding_model=embedding_model,
        vector_store=chroma_vector_store,
    )

    index = builder.build(nodes=nodes)

    total_vectors = chroma_vector_store._collection.count()

    print(
        f"✅ Index built successfully. "
        f"Stored vectors: {total_vectors}"
    )


# ============================================================
# 6. Initialize Retrieval and LLM components
#    Configure retriever and language model for RAG generation
# ============================================================

print("\n" + "=" * 70)
print("🔍 Initializing Retrieval and LLM Components")
print("=" * 70)

similarity_top_k = config.similarity_top_k

retriever = Retriever(
    index,
    similarity_top_k = similarity_top_k,
)

print("✅ Retriever initialized (top_k=3).")


deepseek = DeepSeekLLM(
    model=config.llm_model,
    temperature=config.temperature,
)

llm = deepseek.get_llm()

print(
    f"✅ LLM initialized: {type(llm).__name__}"
)


# ============================================================
# 7. Assemble RAG pipeline
#    Combine retrieval and generation components
# ============================================================

print("\n" + "=" * 70)
print("🚀 Building Basic RAG Pipeline")
print("=" * 70)


rag = BasicRAG(
    retriever=retriever,
    llm=llm,
)

print("✅ BasicRAG pipeline assembled successfully.")


# ============================================================
# 8. Query the RAG system
# ============================================================

print("\n" + "=" * 70)
print("💬 Querying FUTURE-RAG")
print("=" * 70)


question = "What are the main principles of the FUTURE-AI guideline?"

print(f"❓ Question: {question}")

response = rag.query(question)


# ============================================================
# 9. Display final answer
# ============================================================

print("\n" + "=" * 70)
print("🤖 RAG Response")
print("=" * 70)

print(response.response)

print("\n✅ FUTURE-RAG pipeline execution completed successfully!")

📄 Loading Documents
📂 Source path: d:\Future AI Workspace\future-rag\data\knowledge\pdf
✅ Loaded 22 document(s).

✂️  Chunking Documents
⚙️ Chunk size   : 1024
⚙️ Chunk overlap: 256
✅ Generated 60 chunk(s)/node(s).

🧠 Initializing Embedding Model


2026-07-29 19:27:12,345 - INFO - Loading SentenceTransformer model from BAAI/bge-small-en-v1.5.
2026-07-29 19:27:14,750 - INFO - Loaded 1 prompt with these keys: ['query']


✅ Embedding model ready: HuggingFaceEmbedding

🗄️  Connecting to ChromaDB
📂 Database path : d:\Future AI Workspace\future-rag\data\vectorstores\chroma
📌 Collection    : future_rag
✅ Connected to ChromaVectorStore successfully!

📚 Preparing Vector Index
ℹ️ No existing vectors found.
🏗️ Building new index using IndexBuilder...
✅ Index built successfully. Stored vectors: 60

🔍 Initializing Retrieval and LLM Components
✅ Retriever initialized (top_k=3).
✅ LLM initialized: OpenAILike

🚀 Building Basic RAG Pipeline
✅ BasicRAG pipeline assembled successfully.

💬 Querying FUTURE-RAG
❓ Question: What are the main principles of the FUTURE-AI guideline?


2026-07-29 19:27:26,117 - INFO - HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"



🤖 RAG Response
Based solely on the provided context, the main principles of the FUTURE-AI guideline are fairness, universality, traceability, usability, robustness, and explainability.

✅ FUTURE-RAG pipeline execution completed successfully!
